<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Stability008.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# StabilitySweep001 — Act XXXVIII: The Emergent Harmonic

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, fftshift
from tqdm import tqdm

# --- Parameters ---
alpha, beta, gamma, delta = 0.1, 0.01, 0.2, 0.02
T_max = 500
L_values = [50, 75, 100, 125, 150]
seed = 42

# --- Laplacian Function ---
def laplacian(Phi):
    return (
        np.roll(Phi, 1, axis=0) + np.roll(Phi, -1, axis=0) +
        np.roll(Phi, 1, axis=1) + np.roll(Phi, -1, axis=1) -
        4 * Phi
    )

# --- Lambda Extraction ---
def extract_lambda_dom(Phi, L):
    spectrum = np.abs(fftshift(fft2(Phi)))**2
    center = np.array(spectrum.shape) // 2
    spectrum[center[0], center[1]] = 0
    peak_idx = np.unravel_index(np.argmax(spectrum), spectrum.shape)
    kx = peak_idx[1] - center[1]
    ky = peak_idx[0] - center[0]
    k_dom = np.sqrt(kx**2 + ky**2) / L
    return 1 / k_dom if k_dom != 0 else np.inf

# --- Run L-scan ---
results = []
for L in L_values:
    np.random.seed(seed)
    Phi = np.random.randn(L, L)
    for t in tqdm(range(T_max), desc=f"L={L}"):
        Phi += alpha * laplacian(Phi) - beta * Phi + gamma * np.tanh(Phi) + delta * np.random.randn(L, L)
    lambda_dom = extract_lambda_dom(Phi, L)
    results.append((L, lambda_dom))

# --- Display Results ---
print("\nFinal Table of Results")
print("| L (Lattice Size) | λ_dom (Dominant Wavelength) | Ratio (λ_dom / L) |")
print("|------------------|-----------------------------|-------------------|")
for L, lam in results:
    ratio = lam / L
    print(f"| {L:<16} | {lam:>27.2f} | {ratio:>17.3f} |")


L=150: 100%|██████████| 500/500 [00:01<00:00, 405.34it/s]


Final Table of Results
| L (Lattice Size) | λ_dom (Dominant Wavelength) | Ratio (λ_dom / L) |
|------------------|-----------------------------|-------------------|
| 50               |                       50.00 |             1.000 |
| 75               |                       75.00 |             1.000 |
| 100              |                      100.00 |             1.000 |
| 125              |                      125.00 |             1.000 |
| 150              |                       75.00 |             0.500 |
